In [ ]:
import pandas as pd
import glob
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf

In [ ]:
csv_files = glob.glob("data/nba_shots/*.csv")

shots = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

In [ ]:
# print(shots.ZONE_NAME.unique())
# shots.ZONE_RANGE.unique()
# shots.SHOT_TYPE.unique()
# shots.EVENT_TYPE.unique()
# shots.ACTION_TYPE.unique()
# print(shots.LOC_X.describe())
# shots.LOC_Y.describe()
# shots.SHOT_DISTANCE.describe()
# shots.BASIC_ZONE.unique()
shots.QUARTER.value_counts()

In [ ]:
shots.columns

In [ ]:
shots.SHOT_MADE

In [ ]:
# Predictors - for sure


# Predictors - maybe
# 'SHOT_TYPE', 'BASIC_ZONE', 'ZONE_NAME', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'SHOT_DISTANCE', 'QUARTER', 
# 'MINS_LEFT', 'SECS_LEFT', 'ACTION_TYPE'

# Variables measuring the same thing in different ways
# - Distance
#   'SHOT_DISTANCE', 'ZONE_RANGE', 'LOC_X', 'LOC_Y', 'BASIC_ZONE', 'SHOT_TYPE'
#   - Can start with just distance

# - Location
#   'BASIC_ZONE', 'ZONE_NAME', 'LOC_X', 'LOC_Y'
#   - Could use angle
#   - And left/right/center

# - Time
#   'QUARTER', 'MINS_LEFT', 'SECS_LEFT'
#   - Interaction between quarter and seconds left in the quarter

# - Shot Type
#   'ACTION_TYPE', 'SHOT_TYPE'
#   - Want to use action type, it will just need some data cleaning (probably the next iteration of the model)

shots = shots[shots.QUARTER < 8]

shots['right'] = shots.LOC_X < 0
shots['left'] = shots.LOC_X > 0
shots['angle'] = abs(90 - np.degrees(np.arctan2(shots.LOC_Y, shots.LOC_X)))
shots['quarter_1'] = shots.QUARTER == 1
shots['quarter_2'] = shots.QUARTER == 2
shots['quarter_3'] = shots.QUARTER == 3
shots['quarter_4'] = shots.QUARTER == 4
shots['quarter_5'] = shots.QUARTER == 5
shots['quarter_6'] = shots.QUARTER == 6
shots['quarter_7'] = shots.QUARTER == 7
shots['total_secs_left'] = shots.MINS_LEFT * 60 + shots.SECS_LEFT
shots['shots_made'] = shots.SHOT_MADE.astype(int)


shots_X = shots[['SHOT_DISTANCE', 'right', 'left', 'angle', 'quarter_2', 'quarter_3', 'quarter_4',
                 'quarter_5', 'quarter_6', 'quarter_7', 'total_secs_left']]
shots_y = shots.SHOT_MADE.astype(int)

scaler = StandardScaler()

# Select numeric columns
numeric_cols = ['SHOT_DISTANCE', 'angle', 'total_secs_left']

# Fit and transform the numeric columns
shots[['shot_distance_std', 'angle_std', 'total_secs_left_std']] = scaler.fit_transform(shots[numeric_cols])

In [ ]:
shots

In [ ]:
model = smf.ols(formula="shots_made ~ shot_distance_std + angle_std + quarter_2 + quarter_3 + quarter_4 + quarter_5 + quarter_6 + quarter_7 + total_secs_left_std", data=shots).fit()

# View the comprehensive R-style summary table
print(model.summary())

Quarter effect is interesting, distance and seconds make sense, I'm confused on the angle effect though.

In [ ]:
# X = shots_X
# y = shots_y

# lm = LinearRegression()
# lm.fit(X, y)

# print(lm.intercept_)
# print(lm.coef_)

In [ ]:
# y_pred = lm.predict(X)

# display(X)
# y_pred